# 말뭉치를 다른 임베딩 모델로 인코딩 (Colab)

**하는 일은 하나입니다** — 논문 37,854편을 고른 모델로 인코딩해 `.npy` 하나를 만듭니다.
판정은 여기서 하지 않습니다. 행렬을 저장소로 가져가면 기존 하네스가 잽니다.

## 넘어야 하는 바

미래 라벨 과제의 **간선 0개 집단**(1,112 시드, 그래프가 0점이라 임베딩이 유일한 신호)에서
`text-embedding-3-large`를 **구간이 0을 제외하도록** 이겨야 합니다. 같은 자리에서 작은
형제 `bge-base-en-v1.5`가 그 선 아래로 측정됐습니다(recall@5 −0.0102 [−0.0159, −0.0048]).
빈 바가 아닙니다 — ADR-039.

| 모델 | 크기 | 차원 | 여기서 재는가 |
|---|---:|---:|---|
| `BAAI/bge-m3` | 568M | 1024 | 예 |
| `Qwen/Qwen3-Embedding-0.6B` | 595M | 1024 | 예 |
| `Qwen/Qwen3-Embedding-8B` | 8B | 4096 | **아니오** — fp16 16GB라 배포 경로가 없습니다 |

## 하지 않는 것

**텍스트를 다시 만들지 않습니다.** 순서와 문자열이 `corpus.jsonl.gz`에 굳어 있고, 다시
만들면 재는 것이 모델이 아니라 **전처리 차이**가 됩니다.

**Aura를 건드리지 않습니다.** 비교는 파생 캐시에서 돌아가므로 인덱스 재생성도 비용도 없습니다.


## 1. 설정 — 여기만 바꿉니다


In [ ]:
# 이 셀만 바꾸고 나머지는 그대로 실행하세요.
MODEL = "Qwen/Qwen3-Embedding-0.6B"   # 또는 "BAAI/bge-m3" (이미 쟀습니다)

# 2026-10-20 실측으로 올렸습니다. BGE-M3(568M)를 배치 16으로 돌렸을 때 VRAM 피크가
# 조각 19개 전부 **2.7GB로 평평**했습니다 — T4 15.6GB에서 12.9GB가 놀았습니다.
# 가중치 2.27GB를 빼면 활성값이 배치 16에 0.43GB, 즉 **표본당 약 27MB**입니다.
#
# Qwen3-0.6B는 층이 더 많아(28 대 24) 표본당 더 쓸 수 있으므로 45MB로 보수적으로 잡으면
# 배치 128에서 피크가 2.4 + 128x0.045 = 약 8.2GB입니다. 15.6GB에 여유가 남습니다.
# **빗나가도 안전합니다** - OOM이 나면 아래 `encode()`가 절반씩 줄입니다(128 -> 64 -> 32).
#
# 속도는 배치에 비례하지 않습니다. 512토큰 배치 16으로도 T4가 어느 정도 차 있었으므로
# 8배가 아니라 1.5~2.5배를 기대하세요.
BATCH = 128

# 입력을 512토큰에서 자릅니다. 이 말뭉치의 제목+초록 토큰 분포가 중앙값 258이고
# **512에서 초과 논문이 0.0%**입니다(advanced.md). BGE-M3(8192)와 Qwen3(32k)의
# 기본 한도를 그대로 쓰면 어텐션 활성값이 seq_len^2으로 커져 OOM 위험만 늘고,
# 잘리는 논문은 한 편도 없으므로 얻는 것이 없습니다. 기존 비교(512)와도 같은
# 조건이 됩니다.
MAX_SEQ = 512

# 말뭉치를 어디서 받을지. HF 데이터셋 저장소에 올려 뒀다면 USE_HF=True로 두고,
# 아니면 왼쪽 파일 창에 corpus.jsonl.gz를 직접 올리세요(18MB).
#
# **VS Code로 붙은 세션에서는 USE_HF=True가 안 됩니다** - Colab 시크릿 조회가
# "Secrets can only be fetched when running from the Colab UI"로 시간 초과됩니다.
# 그 경우 파일 창에 직접 올리세요.
USE_HF = False
DATA_REPO = "arxiv-graphrag-colab"   # 6절 업로드에도 씁니다
CORPUS = "corpus.jsonl.gz"

# SMOKE=True면 200편만 인코딩해 코드가 도는지 봅니다(CPU로도 1분).
SMOKE = False


## 2. 환경 확인

GPU가 없으면 멈춥니다. CPU로는 이 크기 모델이 10시간을 넘겨서, 시작하는 것 자체가
낭비입니다(이 프로젝트 PC에서 109M 모델이 CPU로 3.5시간이었습니다).


In [ ]:
try:
    import torch
except ImportError:
    torch = None

if torch is None or not torch.cuda.is_available():
    print('GPU가 없습니다. 런타임 > 런타임 유형 변경에서 T4를 고르세요.')
    if not SMOKE:
        raise SystemExit('GPU 없이 전체 인코딩을 시작하지 않습니다.')
else:
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'{name}  {total:.1f}GB')
    # 2026-10-20 실측: BGE-M3(가중치 2.27GB)가 배치 16에서 피크 2.7GB.
    # 표본당 약 27MB이므로 아래 어림은 Qwen3를 보수적으로 본 45MB 기준입니다.
    expected = 2.4 + BATCH * 0.045
    print(f'배치 {BATCH}에서 어림한 피크 {expected:.1f}GB '
          f'(가중치 2.4GB + 표본당 45MB). 조각마다 실측을 찍습니다.')
    if expected > total * 0.75:
        suggested = max(1, int((total * 0.75 - 2.4) / 0.045))
        print(f'여유가 빡빡합니다. BATCH를 {suggested} 이하로 두는 쪽이 안전합니다 '
              f'(OOM이 나도 자동으로 절반씩 줄지만, 줄기 전에 한 번 터집니다).')


In [ ]:
%pip -q install -U sentence-transformers

# 설치만으로 끝내지 않고 **실제로 가져와 봅니다.** 2026-10-20에 이 노트북에서
# `huggingface_hub`를 따로 올렸다가 sentence-transformers(hub<2.0 요구)와 충돌했고,
# pip은 경고만 찍고 넘어갔습니다. 그러면 몇 셀 뒤 인코딩에서야 깨집니다.
try:
    import sentence_transformers
    import huggingface_hub
    print('sentence-transformers', sentence_transformers.__version__)
    print('huggingface_hub', huggingface_hub.__version__)
    from sentence_transformers import SentenceTransformer  # noqa: F401
    print('가져오기 확인 완료')
except Exception as exc:
    raise SystemExit(
        f'{type(exc).__name__}: {exc}\n\n'
        '의존성이 어긋났습니다. 런타임 > 세션 다시 시작을 누르고 이 셀부터 다시 돌리세요. '
        '`huggingface_hub`를 따로 올리지 마세요 - 6절이 이미 설치된 것을 씁니다.'
    )


## 3. 말뭉치 받기

저장소에서 `uv run python -m models.embedding_tuning.export_for_colab`(32초)로 만든
`exports/colab/corpus.jsonl.gz`입니다. 37,854행이고 **파생 캐시와 같은 순서**입니다.


In [ ]:
import gzip
import json
from pathlib import Path

path = Path(CORPUS)
if USE_HF and not path.exists():
    from google.colab import userdata
    from huggingface_hub import HfApi, hf_hub_download
    token = userdata.get('HF_TOKEN')
    owner = HfApi(token=token).whoami()['name']
    path = Path(hf_hub_download(f'{owner}/{DATA_REPO}', CORPUS,
                                repo_type='dataset', token=token))

if not path.exists():
    raise SystemExit(f'{CORPUS}가 없습니다. 왼쪽 파일 창에 올리거나 USE_HF=True로 두세요.')

with gzip.open(path, 'rt', encoding='utf-8') as handle:
    rows = [json.loads(line) for line in handle]

if SMOKE:
    rows = rows[:200]

texts = [row['text'] for row in rows]
print(f'{len(texts)}편, 첫 줄: {texts[0][:80]}...')


## 4. 인코딩

`normalize_embeddings=True`와 `float32`는 기존 비교와 **같아야** 합니다
(`models/embedding_tuning/compare_models.py`). 바꾸면 비교가 깨집니다.

BGE가 권하는 질의 접두사는 **쓰지 않습니다** — 그 접두사는 짧은 질의로 긴 문서를 찾는
비대칭 검색용이고, 이 과제는 논문으로 논문을 찾는 대칭 검색입니다. Qwen3-Embedding의
`prompt_name`도 같은 이유로 쓰지 않습니다.

중간에 끊겨도 처음부터 하지 않도록 2,000편씩 조각으로 저장합니다. 다시 실행하면
남은 조각부터 이어서 합니다.

**OOM이 나면 배치를 절반으로 줄여 다시 시도합니다.** 조각 저장과 합쳐지면 터지는 대신
느려지기만 합니다. 조각마다 **VRAM 피크를 찍으니** 첫 줄에서 여유를 바로 볼 수 있습니다.
한 번 줄어든 배치는 **다시 올라가지 않습니다** — 남은 조각도 그 크기로 갑니다.

| 2026-10-20 실측 (T4 15.6GB) | 값 |
|---|---:|
| BGE-M3 가중치 (fp32) | 2.27GB |
| 배치 16에서 VRAM 피크 (조각 19개 전부) | **2.7GB** |
| 표본당 활성값 | 약 27MB |
| 전체 37,854편 소요 | 44.0분 |

**놀던 12.9GB를 쓰려고 배치를 128로 올렸습니다**(1절). 피크 어림이 8.2GB이고, 빗나가도
절반씩 줄어듭니다. 속도는 배치에 비례하지 않습니다 — 512토큰 배치 16으로도 T4가 어느 정도
차 있었으므로 1.5~2.5배를 기대하세요.


In [ ]:
import re
import time

import numpy as np
from sentence_transformers import SentenceTransformer

slug = re.sub(r'[^a-z0-9]+', '-', MODEL.lower()).strip('-')
out = Path(f'local-{slug}.npy')
chunks = Path(f'chunks-{slug}')
chunks.mkdir(exist_ok=True)
STEP = 2000

model = SentenceTransformer(MODEL, trust_remote_code=True)
print('모델 기본 max_seq_length', model.max_seq_length)
model.max_seq_length = MAX_SEQ
print('이번에 쓰는 max_seq_length', model.max_seq_length)


def encode(batch_texts, size):
    """OOM이면 배치를 절반으로 줄여 다시 시도합니다. 1까지 줄여도 안 되면 올립니다."""
    while True:
        try:
            return model.encode(
                batch_texts, batch_size=size, normalize_embeddings=True,
                show_progress_bar=False, convert_to_numpy=True,
            ), size
        except torch.cuda.OutOfMemoryError:
            if size <= 1:
                raise
            torch.cuda.empty_cache()
            size = max(1, size // 2)
            print(f'  OOM - 배치를 {size}로 줄여 다시 시도합니다', flush=True)


start = time.perf_counter()
size = BATCH
for begin in range(0, len(texts), STEP):
    part = chunks / f'{begin:06d}.npy'
    if part.exists():
        continue  # 끊겼다가 다시 돌린 경우
    vectors, size = encode(texts[begin:begin + STEP], size)
    np.save(part, np.asarray(vectors, dtype=np.float32))
    done = min(begin + STEP, len(texts))
    peak = (
        torch.cuda.max_memory_allocated() / 1e9
        if torch is not None and torch.cuda.is_available() else 0.0
    )
    print(f'{done}/{len(texts)}  {(time.perf_counter() - start) / 60:.1f}분  배치 {size}  VRAM 피크 {peak:.1f}GB', flush=True)

matrix = np.concatenate([np.load(part) for part in sorted(chunks.glob('*.npy'))])
print('행렬', matrix.shape, matrix.dtype)


## 5. 저장하고 확인

행 수가 말뭉치와 다르면 저장하지 않습니다 — 어긋난 행렬을 가져가면 **조용히 다른**
**논문의 벡터로** 점수가 나옵니다. 저장소 쪽 `Graph`도 같은 검사를 합니다.


In [ ]:
if matrix.shape[0] != len(texts):
    raise SystemExit(f'행 수가 다릅니다: {matrix.shape[0]} != {len(texts)}')

norms = np.linalg.norm(matrix, axis=1)
print(f'노름 중앙값 {float(np.median(norms)):.4f} (정규화됐으면 1.0)')

np.save(out, matrix)
card = {
    'model': MODEL,
    'rows': int(matrix.shape[0]),
    'dimensions': int(matrix.shape[1]),
    'max_seq_length': int(model.max_seq_length),
    'batch_size_used': int(size),
    'normalize_embeddings': True,
    'minutes': round((time.perf_counter() - start) / 60, 1),
    'corpus': CORPUS,
    'smoke': SMOKE,
}
Path(out.stem + '.json').write_text(json.dumps(card, ensure_ascii=False, indent=2))
print(json.dumps(card, ensure_ascii=False, indent=2))
print(f'{out}  {out.stat().st_size / 1e6:.0f}MB')


## 6. 가져가기 — HF 데이터셋을 거칩니다

**`google.colab.files.download()`를 쓰지 않습니다.** 2026-10-20에 BGE-M3 44분 실행이 끝난 뒤
그 호출이 `Javascript object`만 네 개 찍고 **예외도 없이 아무것도 내려주지 않았습니다** —
VS Code로 붙은 세션에는 그 JS를 실행할 Colab 브라우저 화면이 없기 때문입니다. 셀은 정상
종료로 보이고, 파일이 왔는지는 로컬을 뒤져 봐야 알 수 있었습니다.

같은 이유로 **Colab 시크릿도 읽히지 않습니다**(`Secrets can only be fetched when running from
the Colab UI`). 그래서 토큰은 `getpass`로 받습니다 — 화면에 찍히지 않습니다.

아래 셀이 `.npy`와 `.json`을 **비공개** HF 데이터셋 저장소에 올립니다. 토큰은 **write 권한**이
필요합니다.

받는 쪽은 저장소에서 한 줄입니다. `<repo>`에 셀이 마지막에 찍어 주는 값을 넣으세요
(이 프로젝트에서는 `blanden77/arxiv-graphrag-colab`).

```bash
# BGE-M3 (이미 받았습니다)
hf download blanden77/arxiv-graphrag-colab local-baai-bge-m3.npy local-baai-bge-m3.json --repo-type dataset --local-dir derived/2026-10-14

# Qwen3-0.6B
hf download blanden77/arxiv-graphrag-colab local-qwen-qwen3-embedding-0-6b.npy local-qwen-qwen3-embedding-0-6b.json --repo-type dataset --local-dir derived/2026-10-14
```

`hf`가 없으면 `uv tool install huggingface_hub`로 깔고 `hf auth login`을 한 번 합니다.

그다음 판정은 두 명령입니다. **OpenAI도 Aura도 쓰지 않습니다.**

```bash
uv run python -m evals.related_study --cases evals/cases_related_future.json \
    --variants vector --embeddings derived/2026-10-14/local-<slug>.npy \
    --output reports/embedding_<name>.json

uv run python -m evals.embedding_slice_study \
    --baseline reports/embedding_openai.json --candidate reports/embedding_<name>.json
```

**받기가 막히면 조각 폴더로 갑니다.** `chunks-<slug>/`의 `.npy` 19개(개당 8MB)를 받아
`scripts/assemble_chunks.py`에 넘기면 같은 행렬이 나옵니다 — 그 스크립트가 이름·순서·빠진
조각·차원·행 수를 합치기 전에 검사합니다.


In [ ]:
import getpass
import re
from pathlib import Path

from huggingface_hub import HfApi

# 올릴 파일. `out`이 아니라 `MODEL`에서 다시 만들고 쓰는 것을 **여기서 다시 임포트**합니다 -
# 1절과 이 셀만 돌려도 올라가야 하기 때문입니다(인코딩이 끝난 뒤 커널이 끊기는 일이 있습니다).
_slug = re.sub(r'[^a-z0-9]+', '-', MODEL.lower()).strip('-')
_targets = [p for p in (Path(f'local-{_slug}.npy'), Path(f'local-{_slug}.json')) if p.exists()]
if not _targets:
    raise SystemExit(f'local-{_slug}.npy 가 없습니다. 5절을 먼저 돌리세요.')
for _p in _targets:
    print(f'{_p.name}  {_p.stat().st_size / 1e6:.1f}MB')

# 토큰은 화면에 찍히지 않습니다. Colab 시크릿은 VS Code 세션에서 시간 초과되므로
# 먼저 시도하되 실패하면 직접 받습니다.
_token = None
try:
    from google.colab import userdata
    _token = userdata.get('HF_TOKEN')
except Exception:
    pass
if not _token:
    _token = getpass.getpass('HF 토큰(write 권한 필요): ')

_api = HfApi(token=_token)
_repo = f"{_api.whoami()['name']}/{DATA_REPO}"
_api.create_repo(_repo, repo_type='dataset', private=True, exist_ok=True)
for _p in _targets:
    _api.upload_file(path_or_fileobj=str(_p), path_in_repo=_p.name,
                     repo_id=_repo, repo_type='dataset')
    print('올림:', _p.name)

print(f'\nrepo: {_repo}')
# **명령을 한 줄로 찍습니다.** 받기는 보통 다른 컴퓨터에서 돌고, 그쪽이 PowerShell이면
# bash의 줄바꿈 백슬래시가 **깨집니다** - 2026-10-23에 첫 줄만 실행돼
# --repo-type dataset이 빠지고 "데이터셋을 모델로 찾는" 404가 났습니다.
print('저장소에서 받을 명령(한 줄입니다):')
print(f'  hf download {_repo} {" ".join(p.name for p in _targets)}'
      f' --repo-type dataset --local-dir derived/2026-10-14')
